In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Load the dataset (available via seaborn or from Kaggle)
import seaborn as sns
df = sns.load_dataset('titanic')

# Quick exploration
print(df.shape)
print(df.info())
print(df.head())
print(df['survived'].value_counts(normalize=True))
def preprocess_titanic(df):
    """Clean and prepare features for modeling."""
    data = df.copy()

    # Select relevant columns
    features = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
    data = data[features]

    # Handle missing values
    data['age'].fillna(data['age'].median(), inplace=True)
    data['fare'].fillna(data['fare'].median(), inplace=True)
    data['embarked'].fillna(data['embarked'].mode()[0], inplace=True)

    # Encode categorical variables
    data['sex'] = LabelEncoder().fit_transform(data['sex'])  # male=1, female=0
    data['embarked'] = LabelEncoder().fit_transform(data['embarked'])
    data['alone'] = data['alone'].astype(int)

    # Create family size feature
    data['family_size'] = data['sibsp'] + data['parch'] + 1

    # Create age groups
    data['age_group'] = pd.cut(data['age'], bins=[0, 12, 18, 35, 60, 100],
                                labels=[0, 1, 2, 3, 4]).astype(int)

    return data

# Preprocess
df_clean = preprocess_titanic(df)
print(df_clean.isnull().sum())  # Verify no missing values
# Separate features and target
X = df_clean.drop('survived', axis=1)
y = df_clean['survived']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
scaler = StandardScaler()
numerical_cols = ['age', 'fare', 'family_size']
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
# Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print(classification_report(y_test, lr_pred, target_names=['Died', 'Survived']))

# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(classification_report(y_test, rf_pred, target_names=['Died', 'Survived']))

# Cross-validation for more robust evaluation
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')
print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
# See which features matter most
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance.to_string(index=False))
def predict_survival(model, scaler, passenger_data):
    """
    Predict survival for a new passenger.

    passenger_data: dict with keys matching feature names
    """
    df_new = pd.DataFrame([passenger_data])
    df_new[numerical_cols] = scaler.transform(df_new[numerical_cols])

    prediction = model.predict(df_new)[0]
    probability = model.predict_proba(df_new)[0][1]

    return prediction, probability

# Example: Would a 25-year-old woman in first class survive?
new_passenger = {
    'pclass': 1,
    'sex': 0,          # 0 = female
    'age': 25,
    'sibsp': 0,
    'parch': 0,
    'fare': 100,
    'embarked': 2,     # Southampton
    'alone': 1,
    'family_size': 1,
    'age_group': 2
}

survived, prob = predict_survival(rf_model, scaler, new_passenger)
print(f"\nPrediction: {'Survived' if survived else 'Died'}")
print(f"Survival probability: {prob:.2%}")


(891, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
None
   survived  pclass     sex   age  sibsp  parch     fare embarke